# RLO Paper Figure Generation

This notebook generates all figures for the RLO (Riemannian Lyapunov Optimizer) paper.

## Figures Overview:
1. **Main Results Table** - Classification, LM, Diffusion, LiT
2. **Training Curves** - Loss/Accuracy over epochs
3. **Ablation Studies** - LR sensitivity, batch size, belief coefficient, eta
4. **Component Analysis** - Sign vs Smooth, with/without belief
5. **NAIM Diagnostics** - Residual metrics, fiber contraction visualization

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
from pathlib import Path
import pandas as pd
from collections import defaultdict

# Publication-quality settings
plt.style.use('seaborn-v0_8-whitegrid')
matplotlib.rcParams.update({
    'font.size': 11,
    'font.family': 'serif',
    'axes.labelsize': 12,
    'axes.titlesize': 13,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 10,
    'figure.figsize': (6, 4),
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'axes.grid': True,
    'grid.alpha': 0.3,
})

# Color scheme - distinguishable and colorblind-friendly
COLORS = {
    'adamw': '#1f77b4',      # Blue
    'lion': '#ff7f0e',       # Orange  
    'rlo': '#2ca02c',        # Green
    'rlo_lambda_a': '#d62728',  # Red
    'smooth_lifted_rlo': '#9467bd',  # Purple
    'sgd': '#8c564b',        # Brown
    'adam': '#e377c2',       # Pink
    'rmsprop': '#7f7f7f',    # Gray
}

LABELS = {
    'adamw': 'AdamW',
    'lion': 'Lion',
    'rlo': 'RLO',
    'rlo_lambda_a': 'RLO-ΛA',
    'smooth_lifted_rlo': 'SmoothLiftedRLO',
    'sgd': 'SGD',
    'adam': 'Adam',
    'rmsprop': 'RMSprop',
}

BASE = Path('/blue/wdixon/wang.yixuan/rlo_experiments')
FIG_DIR = BASE / 'figures'
FIG_DIR.mkdir(exist_ok=True)

print(f"Results directory: {BASE}")
print(f"Figures will be saved to: {FIG_DIR}")

## 1. Load All Results

In [ ]:
def load_results(base_path):
    """Load all JSON results from experiment directories."""
    results = {
        'classification': {},
        'lm': {},
        'diffusion': {},
        'lit': {},
        'ablation': {}
    }
    
    # Classification
    cls_dir = base_path / 'classification'
    if cls_dir.exists():
        for f in cls_dir.glob('*_results.json'):
            with open(f) as fp:
                data = json.load(fp)
                key = f"{data.get('model', 'unknown')}_{data.get('opt', 'unknown')}"
                results['classification'][key] = data
    
    # LM
    lm_dir = base_path / 'lm'
    if lm_dir.exists():
        for f in lm_dir.glob('lm_*_results.json'):
            with open(f) as fp:
                data = json.load(fp)
                results['lm'][data.get('opt', 'unknown')] = data
    
    # Diffusion
    diff_dir = base_path / 'diffusion'
    if diff_dir.exists():
        for f in diff_dir.glob('diff_*_results.json'):
            with open(f) as fp:
                data = json.load(fp)
                key = f"{data.get('res', 'unknown')}_{data.get('opt', 'unknown')}"
                results['diffusion'][key] = data
    
    # LiT
    lit_dir = base_path / 'lit'
    if lit_dir.exists():
        for f in lit_dir.glob('lit_*_results.json'):
            with open(f) as fp:
                data = json.load(fp)
                results['lit'][data.get('opt', 'unknown')] = data
    
    # Ablation
    abl_file = base_path / 'ablation' / 'ablation_results.json'
    if abl_file.exists():
        with open(abl_file) as fp:
            results['ablation'] = json.load(fp)
    
    return results

results = load_results(BASE)
print("Loaded results:")
for k, v in results.items():
    print(f"  {k}: {len(v)} entries")

## 2. Main Results Table (Table 1 in paper)

In [ ]:
def create_main_results_table(results):
    """Create the main results table comparing all optimizers."""
    
    # Extract best metrics
    table_data = []
    
    # Classification - ResNet-50 CIFAR-100
    for opt in ['adamw', 'lion', 'rlo', 'rlo_lambda_a', 'smooth_lifted_rlo']:
        key = f'resnet50_{opt}'
        if key in results['classification']:
            acc = results['classification'][key].get('best_acc', 'N/A')
            table_data.append({'Task': 'ResNet-50/CIFAR-100', 'Optimizer': LABELS[opt], 
                             'Metric': 'Acc (%)', 'Value': f"{acc:.2f}" if isinstance(acc, float) else acc})
    
    # Classification - ViT-S/16 ImageNet
    for opt in ['adamw', 'lion', 'rlo', 'rlo_lambda_a', 'smooth_lifted_rlo']:
        key = f'vit_s16_{opt}'
        if key in results['classification']:
            acc = results['classification'][key].get('best_acc', 'N/A')
            table_data.append({'Task': 'ViT-S/16/ImageNet', 'Optimizer': LABELS[opt],
                             'Metric': 'Acc (%)', 'Value': f"{acc:.2f}" if isinstance(acc, float) else acc})
    
    # Classification - ViT-B/16 ImageNet  
    for opt in ['adamw', 'lion', 'rlo', 'rlo_lambda_a', 'smooth_lifted_rlo']:
        key = f'vit_b16_{opt}'
        if key in results['classification']:
            acc = results['classification'][key].get('best_acc', 'N/A')
            table_data.append({'Task': 'ViT-B/16/ImageNet', 'Optimizer': LABELS[opt],
                             'Metric': 'Acc (%)', 'Value': f"{acc:.2f}" if isinstance(acc, float) else acc})
    
    # LM - GPT-2
    for opt in ['adamw', 'lion', 'rlo', 'rlo_lambda_a', 'smooth_lifted_rlo']:
        if opt in results['lm']:
            ppl = results['lm'][opt].get('best_ppl', 'N/A')
            table_data.append({'Task': 'GPT-2/WikiText', 'Optimizer': LABELS[opt],
                             'Metric': 'PPL ↓', 'Value': f"{ppl:.2f}" if isinstance(ppl, float) else ppl})
    
    # Diffusion 64x64
    for opt in ['adamw', 'lion', 'rlo', 'rlo_lambda_a', 'smooth_lifted_rlo']:
        key = f'64_{opt}'
        if key in results['diffusion']:
            fid = results['diffusion'][key].get('best_fid', 'N/A')
            table_data.append({'Task': 'Diffusion 64×64', 'Optimizer': LABELS[opt],
                             'Metric': 'FID ↓', 'Value': f"{fid:.2f}" if isinstance(fid, float) else fid})
    
    # LiT
    for opt in ['adamw', 'lion', 'rlo', 'rlo_lambda_a', 'smooth_lifted_rlo']:
        if opt in results['lit']:
            loss = results['lit'][opt].get('best_loss', 'N/A')
            table_data.append({'Task': 'LiT-B', 'Optimizer': LABELS[opt],
                             'Metric': 'Loss ↓', 'Value': f"{loss:.4f}" if isinstance(loss, float) else loss})
    
    df = pd.DataFrame(table_data)
    return df

df_main = create_main_results_table(results)
print("\n" + "="*80)
print("MAIN RESULTS TABLE")
print("="*80)
if not df_main.empty:
    # Pivot for better display
    pivot = df_main.pivot_table(index='Task', columns='Optimizer', values='Value', aggfunc='first')
    print(pivot.to_string())
    pivot.to_latex(FIG_DIR / 'table1_main_results.tex')
    print(f"\nSaved to {FIG_DIR / 'table1_main_results.tex'}")
else:
    print("No results loaded yet. Run experiments first.")

## 3. Training Curves (Figure 1)

In [ ]:
def plot_training_curves(results, task='classification', model='vit_s16', metric='acc'):
    """Plot training curves for a specific task."""
    fig, ax = plt.subplots(figsize=(7, 5))
    
    optimizers = ['adamw', 'lion', 'rlo', 'rlo_lambda_a', 'smooth_lifted_rlo']
    
    for opt in optimizers:
        key = f'{model}_{opt}' if task == 'classification' else opt
        data = results.get(task, {}).get(key, {})
        hist = data.get('hist', {})
        
        if metric in hist and len(hist[metric]) > 0:
            values = hist[metric]
            epochs = range(1, len(values) + 1)
            ax.plot(epochs, values, label=LABELS[opt], color=COLORS[opt], linewidth=2)
    
    ax.set_xlabel('Epoch')
    ylabel = 'Accuracy (%)' if metric == 'acc' else 'Loss' if metric == 'loss' else 'Perplexity'
    ax.set_ylabel(ylabel)
    ax.legend(loc='best', framealpha=0.9)
    ax.set_title(f'{model.upper()} Training {ylabel}')
    
    plt.tight_layout()
    return fig

# Plot for each model
for model in ['resnet50', 'vit_s16', 'vit_b16']:
    fig = plot_training_curves(results, 'classification', model, 'acc')
    fig.savefig(FIG_DIR / f'fig_training_{model}_acc.pdf')
    fig.savefig(FIG_DIR / f'fig_training_{model}_acc.png')
    plt.show()
    print(f"Saved: fig_training_{model}_acc.pdf")

In [ ]:
def plot_lm_training(results):
    """Plot LM perplexity curves."""
    fig, ax = plt.subplots(figsize=(7, 5))
    
    for opt in ['adamw', 'lion', 'rlo', 'rlo_lambda_a', 'smooth_lifted_rlo']:
        data = results.get('lm', {}).get(opt, {})
        hist = data.get('hist', {})
        
        if 'ppl' in hist and 'step' in hist:
            ppl = hist['ppl']
            # Steps are recorded every 2000 steps for ppl
            steps = [2000 * (i+1) for i in range(len(ppl))]
            ax.plot(steps, ppl, label=LABELS[opt], color=COLORS[opt], linewidth=2)
    
    ax.set_xlabel('Training Steps')
    ax.set_ylabel('Validation Perplexity')
    ax.legend(loc='upper right', framealpha=0.9)
    ax.set_title('GPT-2 Language Modeling on WikiText-103')
    ax.set_yscale('log')
    
    plt.tight_layout()
    fig.savefig(FIG_DIR / 'fig_lm_ppl.pdf')
    fig.savefig(FIG_DIR / 'fig_lm_ppl.png')
    plt.show()
    print("Saved: fig_lm_ppl.pdf")
    return fig

plot_lm_training(results)

## 4. Ablation Studies (Figure 2)

In [ ]:
def plot_lr_sensitivity(results):
    """Plot learning rate sensitivity study."""
    abl = results.get('ablation', {})
    if not abl:
        print("No ablation results found")
        return
    
    fig, ax = plt.subplots(figsize=(7, 5))
    
    lrs = [5e-5, 1e-4, 2e-4, 5e-4, 1e-3]
    
    for opt in ['lion', 'rlo', 'smooth_lifted_rlo']:
        accs = []
        valid_lrs = []
        for lr in lrs:
            key = f'lr_{lr}_{opt}'
            if key in abl:
                accs.append(abl[key])
                valid_lrs.append(lr)
        
        if accs:
            ax.plot(valid_lrs, accs, 'o-', label=LABELS[opt], color=COLORS[opt], 
                   linewidth=2, markersize=8)
    
    ax.set_xscale('log')
    ax.set_xlabel('Learning Rate')
    ax.set_ylabel('Test Accuracy (%)')
    ax.legend(loc='best', framealpha=0.9)
    ax.set_title('Learning Rate Sensitivity (ResNet-50/CIFAR-100)')
    
    plt.tight_layout()
    fig.savefig(FIG_DIR / 'fig_ablation_lr.pdf')
    fig.savefig(FIG_DIR / 'fig_ablation_lr.png')
    plt.show()
    print("Saved: fig_ablation_lr.pdf")

plot_lr_sensitivity(results)

In [ ]:
def plot_belief_coefficient(results):
    """Plot belief coefficient (λ_b) ablation."""
    abl = results.get('ablation', {})
    if not abl:
        print("No ablation results found")
        return
    
    fig, ax = plt.subplots(figsize=(6, 4.5))
    
    lambdas = [0.0, 0.05, 0.1, 0.2, 0.5]
    accs = []
    valid_lambdas = []
    
    for lb in lambdas:
        key = f'belief_{lb}'
        if key in abl:
            accs.append(abl[key])
            valid_lambdas.append(lb)
    
    if accs:
        ax.bar(range(len(valid_lambdas)), accs, color=COLORS['smooth_lifted_rlo'], alpha=0.8)
        ax.set_xticks(range(len(valid_lambdas)))
        ax.set_xticklabels([f'{lb}' for lb in valid_lambdas])
        ax.set_xlabel('Belief Coefficient (λ_b)')
        ax.set_ylabel('Test Accuracy (%)')
        ax.set_title('Effect of Belief Correction Term')
        
        # Highlight optimal
        if accs:
            best_idx = np.argmax(accs)
            ax.bar(best_idx, accs[best_idx], color='gold', alpha=0.9, edgecolor='black', linewidth=2)
    
    plt.tight_layout()
    fig.savefig(FIG_DIR / 'fig_ablation_belief.pdf')
    fig.savefig(FIG_DIR / 'fig_ablation_belief.png')
    plt.show()
    print("Saved: fig_ablation_belief.pdf")

plot_belief_coefficient(results)

In [ ]:
def plot_eta_ablation(results):
    """Plot fiber contraction rate (η) ablation."""
    abl = results.get('ablation', {})
    if not abl:
        print("No ablation results found")
        return
    
    fig, ax = plt.subplots(figsize=(6, 4.5))
    
    etas = [0.1, 0.2, 0.3, 0.5, 1.0]
    accs = []
    valid_etas = []
    
    for eta in etas:
        key = f'eta_{eta}'
        if key in abl:
            accs.append(abl[key])
            valid_etas.append(eta)
    
    if accs:
        ax.plot(valid_etas, accs, 'o-', color=COLORS['smooth_lifted_rlo'], 
               linewidth=2, markersize=10)
        ax.fill_between(valid_etas, accs, alpha=0.2, color=COLORS['smooth_lifted_rlo'])
        ax.set_xlabel('Fiber Contraction Rate (η)')
        ax.set_ylabel('Test Accuracy (%)')
        ax.set_title('Effect of Fiber Contraction Rate')
        
        # Mark optimal range
        ax.axvspan(0.3, 0.5, alpha=0.15, color='green', label='Optimal range')
        ax.legend(loc='best')
    
    plt.tight_layout()
    fig.savefig(FIG_DIR / 'fig_ablation_eta.pdf')
    fig.savefig(FIG_DIR / 'fig_ablation_eta.png')
    plt.show()
    print("Saved: fig_ablation_eta.pdf")

plot_eta_ablation(results)

In [ ]:
def plot_component_analysis(results):
    """Plot component analysis - ablating different parts of RLO."""
    abl = results.get('ablation', {})
    if not abl:
        print("No ablation results found")
        return
    
    fig, ax = plt.subplots(figsize=(8, 5))
    
    components = [
        ('comp_lion_base', 'Lion\n(baseline)'),
        ('comp_rlo_sign', 'RLO\n(sign only)'),
        ('comp_rlo_belief', 'RLO\n(+belief)'),
        ('comp_smooth_no_belief', 'Smooth\n(no belief)'),
        ('comp_smooth_full', 'Smooth\n(full)'),
        ('comp_rlo_lambda_a', 'RLO-ΛA\n(adaptive)'),
    ]
    
    names = []
    accs = []
    colors = []
    
    color_map = {
        'lion_base': COLORS['lion'],
        'rlo_sign': COLORS['rlo'],
        'rlo_belief': COLORS['rlo'],
        'smooth_no_belief': COLORS['smooth_lifted_rlo'],
        'smooth_full': COLORS['smooth_lifted_rlo'],
        'rlo_lambda_a': COLORS['rlo_lambda_a'],
    }
    
    for key, name in components:
        if key in abl:
            names.append(name)
            accs.append(abl[key])
            colors.append(color_map.get(key.replace('comp_', ''), 'gray'))
    
    if accs:
        bars = ax.bar(range(len(names)), accs, color=colors, alpha=0.85, edgecolor='black')
        ax.set_xticks(range(len(names)))
        ax.set_xticklabels(names)
        ax.set_ylabel('Test Accuracy (%)')
        ax.set_title('Component Analysis: Effect of Each RLO Component')
        
        # Add value labels
        for bar, acc in zip(bars, accs):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                   f'{acc:.1f}', ha='center', va='bottom', fontsize=9)
    
    plt.tight_layout()
    fig.savefig(FIG_DIR / 'fig_component_analysis.pdf')
    fig.savefig(FIG_DIR / 'fig_component_analysis.png')
    plt.show()
    print("Saved: fig_component_analysis.pdf")

plot_component_analysis(results)

## 5. Combined Ablation Figure (2x2 subplot)

In [ ]:
def plot_combined_ablation(results):
    """Create a combined 2x2 ablation figure."""
    abl = results.get('ablation', {})
    if not abl:
        print("No ablation results found")
        return
    
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    
    # (a) Learning Rate Sensitivity
    ax = axes[0, 0]
    lrs = [5e-5, 1e-4, 2e-4, 5e-4, 1e-3]
    for opt in ['lion', 'rlo', 'smooth_lifted_rlo']:
        accs = [abl.get(f'lr_{lr}_{opt}', np.nan) for lr in lrs]
        if any(~np.isnan(accs)):
            ax.plot(lrs, accs, 'o-', label=LABELS[opt], color=COLORS[opt], linewidth=2, markersize=8)
    ax.set_xscale('log')
    ax.set_xlabel('Learning Rate')
    ax.set_ylabel('Test Accuracy (%)')
    ax.legend(loc='best')
    ax.set_title('(a) Learning Rate Sensitivity')
    
    # (b) Batch Size Sensitivity
    ax = axes[0, 1]
    batches = [64, 128, 256, 512]
    for opt in ['lion', 'rlo', 'smooth_lifted_rlo']:
        accs = [abl.get(f'batch_{bs}_{opt}', np.nan) for bs in batches]
        if any(~np.isnan(accs)):
            ax.plot(batches, accs, 'o-', label=LABELS[opt], color=COLORS[opt], linewidth=2, markersize=8)
    ax.set_xlabel('Batch Size')
    ax.set_ylabel('Test Accuracy (%)')
    ax.legend(loc='best')
    ax.set_title('(b) Batch Size Sensitivity')
    
    # (c) Belief Coefficient
    ax = axes[1, 0]
    lambdas = [0.0, 0.05, 0.1, 0.2, 0.5]
    accs = [abl.get(f'belief_{lb}', np.nan) for lb in lambdas]
    valid_idx = ~np.isnan(accs)
    if any(valid_idx):
        ax.bar(np.array(range(len(lambdas)))[valid_idx], np.array(accs)[valid_idx], 
              color=COLORS['smooth_lifted_rlo'], alpha=0.8)
        ax.set_xticks(range(len(lambdas)))
        ax.set_xticklabels([f'{lb}' for lb in lambdas])
    ax.set_xlabel('Belief Coefficient (λ_b)')
    ax.set_ylabel('Test Accuracy (%)')
    ax.set_title('(c) Belief Correction Ablation')
    
    # (d) Fiber Contraction Rate
    ax = axes[1, 1]
    etas = [0.1, 0.2, 0.3, 0.5, 1.0]
    accs = [abl.get(f'eta_{eta}', np.nan) for eta in etas]
    valid_idx = ~np.isnan(accs)
    if any(valid_idx):
        ax.plot(np.array(etas)[valid_idx], np.array(accs)[valid_idx], 'o-', 
               color=COLORS['smooth_lifted_rlo'], linewidth=2, markersize=10)
        ax.axvspan(0.3, 0.5, alpha=0.15, color='green', label='Optimal range')
        ax.legend(loc='best')
    ax.set_xlabel('Fiber Contraction Rate (η)')
    ax.set_ylabel('Test Accuracy (%)')
    ax.set_title('(d) Fiber Contraction Rate Ablation')
    
    plt.tight_layout()
    fig.savefig(FIG_DIR / 'fig_ablation_combined.pdf')
    fig.savefig(FIG_DIR / 'fig_ablation_combined.png')
    plt.show()
    print("Saved: fig_ablation_combined.pdf")

plot_combined_ablation(results)

## 6. Bar Chart Comparison (Figure 3)

In [ ]:
def plot_optimizer_comparison_bars(results):
    """Create bar charts comparing optimizers across tasks."""
    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    
    opts = ['adamw', 'lion', 'rlo', 'rlo_lambda_a', 'smooth_lifted_rlo']
    
    # (a) ResNet-50/CIFAR-100
    ax = axes[0]
    accs = [results['classification'].get(f'resnet50_{o}', {}).get('best_acc', 0) for o in opts]
    bars = ax.bar(range(len(opts)), accs, color=[COLORS[o] for o in opts], alpha=0.85)
    ax.set_xticks(range(len(opts)))
    ax.set_xticklabels([LABELS[o] for o in opts], rotation=45, ha='right')
    ax.set_ylabel('Accuracy (%)')
    ax.set_title('ResNet-50/CIFAR-100')
    # Highlight best
    if max(accs) > 0:
        best_idx = np.argmax(accs)
        bars[best_idx].set_edgecolor('gold')
        bars[best_idx].set_linewidth(3)
    
    # (b) ViT-S/16/ImageNet
    ax = axes[1]
    accs = [results['classification'].get(f'vit_s16_{o}', {}).get('best_acc', 0) for o in opts]
    bars = ax.bar(range(len(opts)), accs, color=[COLORS[o] for o in opts], alpha=0.85)
    ax.set_xticks(range(len(opts)))
    ax.set_xticklabels([LABELS[o] for o in opts], rotation=45, ha='right')
    ax.set_ylabel('Accuracy (%)')
    ax.set_title('ViT-S/16/ImageNet')
    if max(accs) > 0:
        best_idx = np.argmax(accs)
        bars[best_idx].set_edgecolor('gold')
        bars[best_idx].set_linewidth(3)
    
    # (c) GPT-2 LM (lower is better)
    ax = axes[2]
    ppls = [results['lm'].get(o, {}).get('best_ppl', float('inf')) for o in opts]
    ppls = [p if p < float('inf') else 0 for p in ppls]
    bars = ax.bar(range(len(opts)), ppls, color=[COLORS[o] for o in opts], alpha=0.85)
    ax.set_xticks(range(len(opts)))
    ax.set_xticklabels([LABELS[o] for o in opts], rotation=45, ha='right')
    ax.set_ylabel('Perplexity ↓')
    ax.set_title('GPT-2/WikiText-103')
    if min([p for p in ppls if p > 0], default=0) > 0:
        best_idx = np.argmin([p if p > 0 else float('inf') for p in ppls])
        bars[best_idx].set_edgecolor('gold')
        bars[best_idx].set_linewidth(3)
    
    # (d) Diffusion FID (lower is better)
    ax = axes[3]
    fids = [results['diffusion'].get(f'64_{o}', {}).get('best_fid', float('inf')) for o in opts]
    fids = [f if f < float('inf') else 0 for f in fids]
    bars = ax.bar(range(len(opts)), fids, color=[COLORS[o] for o in opts], alpha=0.85)
    ax.set_xticks(range(len(opts)))
    ax.set_xticklabels([LABELS[o] for o in opts], rotation=45, ha='right')
    ax.set_ylabel('FID ↓')
    ax.set_title('Diffusion 64×64')
    if min([f for f in fids if f > 0], default=0) > 0:
        best_idx = np.argmin([f if f > 0 else float('inf') for f in fids])
        bars[best_idx].set_edgecolor('gold')
        bars[best_idx].set_linewidth(3)
    
    plt.tight_layout()
    fig.savefig(FIG_DIR / 'fig_comparison_bars.pdf')
    fig.savefig(FIG_DIR / 'fig_comparison_bars.png')
    plt.show()
    print("Saved: fig_comparison_bars.pdf")

plot_optimizer_comparison_bars(results)

## 7. Radar Chart (Spider Plot)

In [ ]:
def plot_radar_chart(results):
    """Create a radar chart comparing optimizers across all tasks."""
    from math import pi
    
    categories = ['ResNet-50', 'ViT-S/16', 'ViT-B/16', 'GPT-2', 'Diffusion', 'LiT']
    N = len(categories)
    
    # Collect normalized scores (higher is better, so invert PPL and FID)
    optimizers = ['adamw', 'lion', 'smooth_lifted_rlo']
    
    fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
    
    angles = [n / float(N) * 2 * pi for n in range(N)]
    angles += angles[:1]
    
    ax.set_theta_offset(pi / 2)
    ax.set_theta_direction(-1)
    
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(categories)
    
    for opt in optimizers:
        values = []
        
        # Normalize each metric to 0-100 scale based on results
        acc1 = results['classification'].get(f'resnet50_{opt}', {}).get('best_acc', 0)
        acc2 = results['classification'].get(f'vit_s16_{opt}', {}).get('best_acc', 0)
        acc3 = results['classification'].get(f'vit_b16_{opt}', {}).get('best_acc', 0)
        ppl = results['lm'].get(opt, {}).get('best_ppl', 1000)
        fid = results['diffusion'].get(f'64_{opt}', {}).get('best_fid', 500)
        lit_loss = results['lit'].get(opt, {}).get('best_loss', 10)
        
        # Normalize (these are rough normalizations)
        values = [
            acc1,  # Already percentage
            acc2,  # Already percentage
            acc3,  # Already percentage
            max(0, 100 - ppl/10),  # Invert PPL
            max(0, 100 - fid),  # Invert FID
            max(0, 100 - lit_loss*10),  # Invert loss
        ]
        
        values += values[:1]
        ax.plot(angles, values, 'o-', linewidth=2, label=LABELS[opt], color=COLORS[opt])
        ax.fill(angles, values, alpha=0.15, color=COLORS[opt])
    
    ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.0))
    ax.set_title('Optimizer Performance Across Tasks', y=1.08)
    
    plt.tight_layout()
    fig.savefig(FIG_DIR / 'fig_radar.pdf')
    fig.savefig(FIG_DIR / 'fig_radar.png')
    plt.show()
    print("Saved: fig_radar.pdf")

plot_radar_chart(results)

## 8. Generate Summary Statistics

In [ ]:
def generate_summary(results):
    """Generate a comprehensive summary of all results."""
    print("\n" + "="*80)
    print("COMPREHENSIVE RESULTS SUMMARY")
    print("="*80)
    
    # Classification
    print("\n### IMAGE CLASSIFICATION ###")
    for model in ['resnet50', 'vit_s16', 'vit_b16']:
        print(f"\n{model.upper()}:")
        model_results = []
        for opt in ['adamw', 'lion', 'rlo', 'rlo_lambda_a', 'smooth_lifted_rlo']:
            key = f'{model}_{opt}'
            acc = results['classification'].get(key, {}).get('best_acc', 'N/A')
            if isinstance(acc, float):
                model_results.append((opt, acc))
                print(f"  {LABELS[opt]:20s}: {acc:.2f}%")
        
        if model_results:
            best = max(model_results, key=lambda x: x[1])
            print(f"  {'BEST':20s}: {LABELS[best[0]]} ({best[1]:.2f}%)")
    
    # LM
    print("\n### LANGUAGE MODELING (GPT-2) ###")
    lm_results = []
    for opt in ['adamw', 'lion', 'rlo', 'rlo_lambda_a', 'smooth_lifted_rlo']:
        ppl = results['lm'].get(opt, {}).get('best_ppl', 'N/A')
        if isinstance(ppl, float):
            lm_results.append((opt, ppl))
            print(f"  {LABELS[opt]:20s}: {ppl:.2f} PPL")
    
    if lm_results:
        best = min(lm_results, key=lambda x: x[1])
        print(f"  {'BEST':20s}: {LABELS[best[0]]} ({best[1]:.2f} PPL)")
    
    # Diffusion
    print("\n### DIFFUSION ###")
    for res in [64, 128]:
        print(f"\n{res}x{res}:")
        diff_results = []
        for opt in ['adamw', 'lion', 'rlo', 'rlo_lambda_a', 'smooth_lifted_rlo']:
            key = f'{res}_{opt}'
            fid = results['diffusion'].get(key, {}).get('best_fid', 'N/A')
            if isinstance(fid, float) and not np.isnan(fid):
                diff_results.append((opt, fid))
                print(f"  {LABELS[opt]:20s}: {fid:.2f} FID")
        
        if diff_results:
            best = min(diff_results, key=lambda x: x[1])
            print(f"  {'BEST':20s}: {LABELS[best[0]]} ({best[1]:.2f} FID)")
    
    # LiT
    print("\n### VISION-LANGUAGE (LiT) ###")
    lit_results = []
    for opt in ['adamw', 'lion', 'rlo', 'rlo_lambda_a', 'smooth_lifted_rlo']:
        loss = results['lit'].get(opt, {}).get('best_loss', 'N/A')
        if isinstance(loss, float):
            lit_results.append((opt, loss))
            print(f"  {LABELS[opt]:20s}: {loss:.4f} Loss")
    
    if lit_results:
        best = min(lit_results, key=lambda x: x[1])
        print(f"  {'BEST':20s}: {LABELS[best[0]]} ({best[1]:.4f} Loss)")
    
    print("\n" + "="*80)

generate_summary(results)

## 9. Export All Figures List

In [ ]:
# List all generated figures
print("\nGenerated figures:")
if FIG_DIR.exists():
    for f in sorted(FIG_DIR.glob('*')):
        print(f"  {f.name}")
else:
    print("  No figures generated yet.")